# 피치클락은 KBO 경기를 얼마나 줄였나? 5,024경기(정규이닝 4,610경기)로 직접 검증하기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/team-datapopcorn/datapopcorn-assets/blob/main/kbo-pitch-clock/kbo_pitch_clock_analysis.ipynb)

KBO는 2024년 피치클락을 시범 운영(경고만)하고 2025년 정식 도입했습니다. 2026년에는 투구 간격을 2초씩 더 줄였는데, 시즌 초부터 "오히려 경기가 길어졌다"는 기사가 나왔습니다.

이 노트북은 KBO 공식 홈페이지의 경기별 기록으로 그 질문을 다시 계산합니다. 블로그 글 [datapopcorn.ai/blog/kbo-pitch-clock-game-time-analysis](https://datapopcorn.ai/blog/kbo-pitch-clock-game-time-analysis)의 모든 숫자와 차트가 이 노트북에서 나옵니다.

**실행 방법**: 메뉴의 `런타임 → 모두 실행`을 누르면 1분 안에 끝납니다. GPU, 로그인, Drive 연결은 필요 없습니다. 기본값은 저장해 둔 스냅샷(2026-10-07 수집)을 읽고, `RECOLLECT = True`로 바꾸면 KBO 홈페이지에서 처음부터 다시 수집합니다(약 10분).

| 데이터 | 출처 | 범위 |
|---|---|---|
| 경기별 소요시간·관중·개시/종료 시각 | KBO 홈페이지 게임센터 스코어보드(`GetScoreBoardScroll`) | 2019, 2021~2026 정규시즌 |
| 경기별 투구 수·상대 타자 수·4사구·삼진 | KBO 홈페이지 박스스코어 투수 기록 합계(`GetBoxScoreScroll`) | 같은 경기 |
| 규칙 변경 시점 | KBO 보도자료·언론 보도 | 2024~2026 |

In [ ]:
# Step 0. 준비: 한글 폰트(Pretendard)를 내려받습니다.
!mkdir -p fonts && cd fonts && for w in Regular Bold ExtraBold; do [ -f Pretendard-$w.otf ] || wget -q https://cdn.jsdelivr.net/gh/orioncactus/pretendard@v1.3.9/packages/pretendard/dist/public/static/Pretendard-$w.otf; done; ls

In [ ]:
import os, io, re, json, gzip, time, tarfile, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

BASE = "https://raw.githubusercontent.com/team-datapopcorn/datapopcorn-assets/main/kbo-pitch-clock/"
LOCAL = os.path.isfile("data/games.csv")          # 같은 폴더에 data/가 있으면 로컬 파일 사용
RECOLLECT = False                                   # True면 KBO 홈페이지에서 다시 수집
OUT = "figures"; os.makedirs(OUT, exist_ok=True)

for f in os.listdir("fonts"):
    if f.endswith(".otf"): font_manager.fontManager.addfont(os.path.join("fonts", f))
plt.rcParams.update({"font.family": "Pretendard", "axes.unicode_minus": False})
BG, INK, SUB, GRID = "#F7F6F1", "#1F2328", "#5B636A", "#DEDBD2"
RED, GRAY, BLUE, GOLD = "#C8415A", "#848D92", "#5B7DB1", "#D99A2B"
def style(ax, title, sub=None):
    ax.set_facecolor(BG); ax.figure.set_facecolor(BG)
    for s in ["top", "right"]: ax.spines[s].set_visible(False)
    for s in ["left", "bottom"]: ax.spines[s].set_color(GRID)
    ax.tick_params(colors=SUB); ax.grid(axis="y", color=GRID, lw=0.8); ax.set_axisbelow(True)
    ax.set_title(title, loc="left", fontsize=16, fontweight="bold", color=INK, pad=24)
    if sub: ax.text(0, 1.02, sub, transform=ax.transAxes, fontsize=10.5, color=SUB)
def hm(m): return f"{int(m//60)}시간 {int(round(m%60))}분"
print("snapshot source:", "local data/" if LOCAL else BASE)

## Step 1. 데이터 불러오기

KBO 홈페이지 게임센터는 경기마다 두 가지 JSON을 돌려줍니다.

- **스코어보드**: 구장, 관중, 개시·종료 시각, `USE_TM`(경기시간)
- **박스스코어**: 양 팀 투수 기록. 표 맨 아래 `TOTAL` 줄에 팀 전체 투구 수, 상대 타자 수, 4사구, 삼진이 있습니다.

아래 셀의 `collect()`가 실제 수집 코드입니다. 기본값(`RECOLLECT = False`)은 같은 코드로 2026-10-07에 받아 둔 원본(`raw_kbo_2019_2026.tar.gz`)과 그것을 정리한 `games.csv`를 읽습니다. 2020 시즌은 무관중·일정 변동이 커서 빼고, 2026 시즌은 수집일까지 끝난 경기만 들어 있습니다.

In [ ]:
API = "https://www.koreabaseball.com/ws/Schedule.asmx/"
HDR = {"User-Agent": "Mozilla/5.0", "X-Requested-With": "XMLHttpRequest",
       "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
       "Referer": "https://www.koreabaseball.com/Schedule/Schedule.aspx"}
def post(fn, data):
    for a in range(4):
        try:
            req = urllib.request.Request(API + fn, data=urllib.parse.urlencode(data).encode(), headers=HDR)
            return json.loads(urllib.request.urlopen(req, timeout=30).read().decode("utf-8"))
        except Exception: time.sleep(1.5 * (a + 1))
    raise RuntimeError(fn)
def game_ids(season):  # 정규시즌(srId=0) 중 '리뷰' 링크가 있는 = 실제로 열린 경기
    ids = set()
    for m in range(3, 12):
        j = post("GetScheduleList", {"leId": 1, "srIdList": "0", "seasonId": season, "gameMonth": f"{m:02d}", "teamId": ""})
        for r in j.get("rows", []):
            for c in r["row"]:
                g = re.search(r"gameId=(\d{8}[A-Z]{4}\d)&section=REVIEW", c.get("Text") or "")
                if g: ids.add(g.group(1))
    return sorted(ids)
def collect(seasons, raw="raw"):
    for s in seasons:
        for gid in game_ids(s):
            p = f"{raw}/{s}/{gid}.json.gz"
            if os.path.exists(p): continue
            os.makedirs(os.path.dirname(p), exist_ok=True)
            d = {"scoreboard": post("GetScoreBoardScroll", {"leId": 1, "srId": 0, "seasonId": s, "gameId": gid}),
                 "boxscore": post("GetBoxScoreScroll", {"leId": 1, "srId": 0, "seasonId": s, "gameId": gid})}
            d["retrieved"] = time.strftime("%Y-%m-%dT%H:%M:%S%z")   # 수집 시각도 함께 저장
            with gzip.open(p, "wt", encoding="utf-8") as f: json.dump(d, f, ensure_ascii=False)
            time.sleep(0.2)

def ip(x):  # '8 2/3' -> 8.667
    return sum(eval(p) if "/" in p else float(p) for p in x.split())
def team_total(side):
    t = json.loads(side["table"]); r = [re.sub("<[^>]+>", "", c["Text"] or "") for c in t["tfoot"][0]["row"]]
    return dict(ip=ip(r[1]), bf=int(r[2]), np=int(r[3]), h=int(r[5]), bb=int(r[7]), so=int(r[8]), r=int(r[9]))
def parse(raw="raw"):
    rows = []
    for root, _, files in os.walk(raw):
        for fn in sorted(files):
            if not fn.endswith(".json.gz") or fn.startswith("._"): continue
            d = json.load(gzip.open(os.path.join(root, fn), "rt", encoding="utf-8")); sb, bx = d["scoreboard"], d["boxscore"]
            if not sb.get("USE_TM") or not bx.get("arrPitcher"): continue
            a, b = (team_total(s) for s in bx["arrPitcher"]); h, m = sb["USE_TM"].split(":")
            rows.append(dict(game_id=sb["G_ID"], date=sb["G_DT"], season=sb["SEASON_ID"], away=sb["AWAY_ID"], home=sb["HOME_ID"],
                             stadium=sb["S_NM"], crowd=int((sb.get("CROWD_CN") or "0").replace(",", "") or 0),
                             minutes=int(h) * 60 + int(m), innings=int(bx["realMaxInning"]),
                             pitches=a["np"] + b["np"], batters=a["bf"] + b["bf"], bb_hbp=a["bb"] + b["bb"],
                             so=a["so"] + b["so"], runs=a["r"] + b["r"], hits=a["h"] + b["h"], outs_ip=a["ip"] + b["ip"]))
    return pd.DataFrame(rows).sort_values("game_id").reset_index(drop=True)

if RECOLLECT:
    collect([2019, 2021, 2022, 2023, 2024, 2025, 2026]); games = parse("raw")
else:
    if not LOCAL:   # 원본 스냅샷을 받아 직접 다시 파싱 -> games.csv와 같은지 확인
        urllib.request.urlretrieve(BASE + "data/raw_kbo_2019_2026.tar.gz", "raw.tar.gz")
        urllib.request.urlretrieve(BASE + "data/games.csv", "games_snapshot.csv")
        snap = pd.read_csv("games_snapshot.csv")
    else:
        os.system("cp data/raw_kbo_2019_2026.tar.gz raw.tar.gz"); snap = pd.read_csv("data/games.csv")
    tarfile.open("raw.tar.gz").extractall(".", filter="data")
    games = parse("raw")
    same = games[snap.columns.intersection(games.columns)].reset_index(drop=True).equals(
        snap.sort_values("game_id")[snap.columns.intersection(games.columns)].reset_index(drop=True))
    print("원본 재파싱 결과 == games.csv :", same)
print(games.shape); games.head()

## Step 2. 검증: 경기 수와 KBO 공식 발표치 맞춰 보기

분석 전에 두 가지를 확인합니다.

1. 시즌마다 정규시즌 720경기가 다 있는가 (2026은 수집일까지)
2. 우리가 계산한 평균 경기시간이 KBO 발표치와 같은가. KBO는 2025시즌 **정규이닝(9이닝) 평균 3시간 2분, 연장 포함 3시간 5분**이라고 발표했습니다.

정규이닝 경기는 `realMaxInning == 9`이면서 두 팀 투수 이닝 합이 17 이상인 경기(9회말 공격을 생략하면 8+9=17)로 정의합니다. 우천 콜드(5~8회)와 연장(10~12회)은 뺍니다. 아래 표의 '전체 평균'은 연장과 우천 콜드까지 모두 넣은 값이라, 우천 콜드 6경기를 빼면 2025년은 185.8분(반올림 3시간 6분, 버림 3시간 5분)입니다.

In [ ]:
games["reg9"] = (games.innings == 9) & (games.outs_ip >= 17)
chk = games.groupby("season").agg(경기수=("game_id", "size"), 정규이닝경기=("reg9", "sum"), 전체평균분=("minutes", "mean"))
chk["정규이닝평균분"] = games[games.reg9].groupby("season")["minutes"].mean()
chk["정규이닝평균"] = chk["정규이닝평균분"].map(hm); chk["전체평균(연장·콜드 포함)"] = chk["전체평균분"].map(hm)
print(chk.round(1).to_string())
g9 = games[games.reg9].copy()

## Step 3. 시즌별 평균 경기시간

먼저 가장 단순한 그림입니다. 2019~2023년은 3시간 7~14분 사이에서 움직이다가 2024년부터 줄고, 2025년 3시간 2분까지 내려갑니다. 2026년은 다시 약 3.5분 늘었습니다(분 단위 반올림 때문에 3:02→3:06으로 보입니다).

In [ ]:
S = sorted(g9.season.unique())
m = g9.groupby("season").agg(dur=("minutes", "mean"), N=("pitches", "mean"), bf=("batters", "mean"),
                              bb=("bb_hbp", "mean"), so=("so", "mean"))
m["spp"] = m.dur * 60 / m.N           # 공 1개당 걸린 시간(초) = 경기시간 / 투구 수 (교대 등 고정시간 포함)
fig, ax = plt.subplots(figsize=(9, 5))
col = [GRAY if s < 2024 else (GOLD if s == 2024 else (RED if s == 2025 else BLUE)) for s in m.index]
ax.bar([str(s) for s in m.index], m.dur, color=col, width=0.62)
for i, (s, v) in enumerate(m.dur.items()): ax.text(i, v + 0.6, f"{int(v//60)}:{int(round(v%60)):02d}", ha="center", color=INK, fontsize=12, fontweight="bold")
ax.set_ylim(170, 200); ax.set_ylabel("분", color=SUB)
style(ax, "KBO 정규이닝(9이닝) 평균 경기시간", "회색 = 규칙 변경 전, 노랑 = 2024 ABS+피치클락 시범, 빨강 = 2025 정식 도입, 파랑 = 2026 2초 단축")
plt.tight_layout(); plt.savefig(f"{OUT}/01_season_minutes.png", dpi=160); plt.show()

## Step 4. 경기시간 = 투구 수 × 공 1개당 시간

경기시간은 두 가지 이유로 바뀝니다.

- **공이 많아져서** (볼넷·풀카운트가 늘면 투구 수가 늘어남)
- **투구 1개당 평균 소요시간이 바뀌어서**

그래서 경기시간을 `투구 수 × 투구 1개당 평균 소요시간`으로 나눠 봅니다. 주의할 점이 있습니다. 투구 1개당 평균 소요시간은 경기 전체 시간을 투구 수로 나눈 값이라 실제 투구 간격(피치클락 20초)과 다릅니다. 이닝 교대, 투수 교체, 비디오 판독, 마운드 방문 시간이 모두 섞여 있고, 투구 수가 늘면 고정 시간이 더 많은 공에 나눠지면서 저절로 조금 내려갑니다. 피치클락이 줄이려는 시간이 이 안에 들어 있지만, 이 값 자체를 피치클락 효과라고 부를 수는 없습니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, col_, title, unit in [(axes[0], "N", "경기당 투구 수(양 팀 합)", "개"), (axes[1], "spp", "투구 1개당 평균 소요시간(교대·교체 포함)", "초")]:
    ax.plot([str(s) for s in m.index], m[col_], marker="o", color=RED if col_ == "spp" else BLUE, lw=2.5)
    for i, v in enumerate(m[col_]): ax.text(i, v + (0.25 if col_ == "spp" else 1.2), f"{v:.1f}", ha="center", color=INK, fontsize=10.5)
    style(ax, title); ax.set_ylabel(unit, color=SUB)
axes[0].set_ylim(285, 315); axes[1].set_ylim(35.5, 39.5)
plt.tight_layout(); plt.savefig(f"{OUT}/02_pitches_and_pace.png", dpi=160); plt.show()
print(m.round(2).to_string())

## Step 5. 변화량 분해: "공이 많아서" vs "공 1개당 시간이 바뀌어서"

두 시즌 사이의 평균 경기시간 변화를 정확히 두 조각으로 나눕니다 (셰플리 분해).

- 투구 수 몫 = ΔN × (두 시즌 투구 1개당 소요시간 평균)
- 1개당 시간 몫 = Δ(투구 1개당 소요시간) × (두 시즌 투구 수 평균)

두 조각을 더하면 실제 변화와 정확히 같습니다. 2025→2026은 경기 단위로 2,000번 다시 뽑아(부트스트랩) 95% 구간도 구합니다.

In [ ]:
def decomp(a, c):
    dN, dr = m.N[c] - m.N[a], m.dur[c] / m.N[c] - m.dur[a] / m.N[a]
    vol = dN * (m.dur[a] / m.N[a] + m.dur[c] / m.N[c]) / 2
    pace = dr * (m.N[a] + m.N[c]) / 2
    return dict(구간=f"{a}→{c}", 총변화=m.dur[c] - m.dur[a], 투구수몫=vol, 개당시간몫=pace, 투구수변화=dN, 볼넷사구변화=m.bb[c] - m.bb[a])
dec = pd.DataFrame([decomp(2023, 2024), decomp(2024, 2025), decomp(2025, 2026), decomp(2023, 2025), decomp(2023, 2026)])
print(dec.round(2).to_string(index=False))

def decomp_df(A, C):
    TA, NA, TC, NC = A.minutes.mean(), A.pitches.mean(), C.minutes.mean(), C.pitches.mean()
    vol = (NC - NA) * (TA / NA + TC / NC) / 2; pace = (TC / NC - TA / NA) * (NA + NC) / 2
    return TC - TA, vol, pace
rng = np.random.default_rng(2026); A25, A26 = g9[g9.season == 2025], g9[g9.season == 2026]
boot = np.array([decomp_df(A25.sample(len(A25), replace=True, random_state=rng.integers(1e9)),
                           A26.sample(len(A26), replace=True, random_state=rng.integers(1e9))) for _ in range(2000)])
lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
for i, name in enumerate(["총변화", "투구수몫", "개당시간몫"]):
    print(f"2025→2026 {name}: {decomp_df(A25, A26)[i]:+.2f}분  95% 구간 [{lo[i]:+.2f}, {hi[i]:+.2f}]")
boot_ci = {n: [round(float(lo[i]), 2), round(float(hi[i]), 2)] for i, n in enumerate(["total", "volume", "pace"])}

# 늘어난 투구는 어디서 왔나: 타석당 투구 수 vs 타석 수
pa = g9.groupby("season").agg(타석=("batters", "mean"), 안타=("hits", "mean"),
                              득점=("runs", "mean"), 볼넷사구=("bb_hbp", "mean"), 삼진=("so", "mean"))
pa["타석당투구"] = m.N / pa.타석
print(pa.loc[[2024, 2025, 2026]].round(3).to_string())

fig, ax = plt.subplots(figsize=(9, 5))
sub = dec.iloc[:3]; x = np.arange(len(sub)); w = 0.36
ax.bar(x - w / 2, sub.투구수몫, w, color=BLUE, label="투구 수가 바뀐 몫")
ax.bar(x + w / 2, sub.개당시간몫, w, color=RED, label="투구 1개당 소요시간이 바뀐 몫")
for i, r in sub.iterrows():
    ax.text(i - w / 2, r.투구수몫 + (0.25 if r.투구수몫 > 0 else -0.6), f"{r.투구수몫:+.1f}분", ha="center", color=INK, fontsize=11)
    ax.text(i + w / 2, r.개당시간몫 + (0.25 if r.개당시간몫 > 0 else -0.6), f"{r.개당시간몫:+.1f}분", ha="center", color=INK, fontsize=11)
    ax.text(i, 5.2, f"합계 {r.총변화:+.1f}분", ha="center", color=INK, fontsize=12, fontweight="bold")
ax.axhline(0, color=SUB, lw=1); ax.set_xticks(x, ["2023→2024\nABS·시범운영", "2024→2025\n피치클락 정식", "2025→2026\n2초 단축"])
ax.set_ylim(-7.5, 6.2); ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
style(ax, "경기시간 변화, 어디서 왔나", "정규이닝 평균 기준, 분")
plt.tight_layout(); plt.savefig(f"{OUT}/03_decomposition.png", dpi=160); plt.show()

## Step 6. 회귀로 한 번 더: 투구 수와 타자 수를 같게 놓고 비교

분해는 평균끼리 비교합니다. 경기마다 투구 수와 상대 타자 수가 다르니, 이번에는 경기 단위 회귀로 **같은 투구 수·같은 타자 수일 때** 시즌별로 몇 분 차이가 나는지 봅니다.

`경기시간(분) = 시즌 효과 + β₁ × 투구 수 + β₂ × 상대 타자 수 + 오차`

기준 시즌은 2023(마지막 규칙 변경 전 시즌)입니다. 표준오차는 이분산에 강건한 HC1으로 계산합니다(numpy로 직접 구현, 추가 패키지 불필요).

In [ ]:
def ols_hc1(X, y, full=False):
    XtXi = np.linalg.inv(X.T @ X); b = XtXi @ X.T @ y; e = y - X @ b; n, k = X.shape
    V = XtXi @ (X.T * e**2) @ X @ XtXi * n / (n - k); return (b, V) if full else (b, np.sqrt(np.diag(V)))
others = [s for s in S if s != 2023]
X = np.column_stack([np.ones(len(g9))] + [(g9.season == s).astype(float) for s in others] + [g9.pitches, g9.batters])
b, se = ols_hc1(X, g9.minutes.values.astype(float))
reg = pd.DataFrame({"시즌": others, "2023 대비(분)": b[1:1 + len(others)], "95% 하한": b[1:1 + len(others)] - 1.96 * se[1:1 + len(others)],
                    "95% 상한": b[1:1 + len(others)] + 1.96 * se[1:1 + len(others)]})
print(reg.round(2).to_string(index=False))
print(f"투구 1개 추가 = {b[-2]*60:.1f}초, 상대 타자 1명 추가 = {b[-1]*60:.1f}초 (같은 시즌, 같은 투구 수 기준)")
bf_, V_ = ols_hc1(X, g9.minutes.values.astype(float), full=True)
i25, i26 = 1 + others.index(2025), 1 + others.index(2026)
d_, sd_ = bf_[i26] - bf_[i25], np.sqrt(V_[i26, i26] + V_[i25, i25] - 2 * V_[i26, i25])
print(f"같은 투구 수·타자 수일 때 2026 - 2025 = {d_:+.2f}분, 95% 구간 [{d_-1.96*sd_:+.2f}, {d_+1.96*sd_:+.2f}]")
reg_2526 = [round(float(d_), 2), round(float(d_ - 1.96 * sd_), 2), round(float(d_ + 1.96 * sd_), 2)]

# 민감도: 2026 수집일까지와 같은 날짜 구간(월-일)만 비교
md_ = g9.date.str[5:]; lastmd = g9[g9.season == 2026].date.str[5:].max(); firstmd = g9[g9.season == 2026].date.str[5:].min()
w_ = g9[(md_ >= firstmd) & (md_ <= lastmd)]
print("같은 날짜 구간", firstmd, "~", lastmd, ": 2025→2026 평균 변화",
      round(w_[w_.season == 2026].minutes.mean() - w_[w_.season == 2025].minutes.mean(), 2), "분")

fig, ax = plt.subplots(figsize=(9, 5))
rr = pd.concat([reg, pd.DataFrame([{"시즌": 2023, "2023 대비(분)": 0, "95% 하한": 0, "95% 상한": 0}])]).sort_values("시즌")
cols = [GRAY if s < 2024 else (GOLD if s == 2024 else (RED if s == 2025 else BLUE)) for s in rr.시즌]
ax.errorbar([str(s) for s in rr.시즌], rr["2023 대비(분)"], yerr=[rr["2023 대비(분)"] - rr["95% 하한"], rr["95% 상한"] - rr["2023 대비(분)"]],
            fmt="none", ecolor=GRID, elinewidth=6)
ax.scatter([str(s) for s in rr.시즌], rr["2023 대비(분)"], c=cols, s=110, zorder=3)
for i, (s_, v) in enumerate(zip(rr.시즌, rr["2023 대비(분)"])): ax.text(i + 0.12, v, "기준" if s_ == 2023 else f"{v:+.1f}분", va="center", color=INK, fontsize=11)
ax.axhline(0, color=SUB, lw=1)
style(ax, "투구 수·타자 수가 같다면, 경기는 몇 분 달라졌나", "2023년 대비, 막대 = 95% 신뢰구간 (HC1)")
plt.tight_layout(); plt.savefig(f"{OUT}/04_regression.png", dpi=160); plt.show()

## Step 7. 2026년은 월별로 어땠나

시즌 초 "경기가 10분 늘었다"는 기사는 3~4월 몇 경기 기준이었습니다. 월별로 나눠 보면 어느 시점에 길어졌는지, 그때 투구 수도 같이 늘었는지 보입니다. 3월은 개막 직후 경기 수가 적으니 숫자를 조심해서 읽어야 합니다.

In [ ]:
g9["month"] = g9.date.str[5:7].astype(int)
mm = g9[g9.season >= 2025].groupby(["season", "month"]).agg(경기수=("minutes", "size"), 평균분=("minutes", "mean"), 투구수=("pitches", "mean")).reset_index()
print(mm.round(1).to_string(index=False))
fig, ax = plt.subplots(figsize=(9, 5))
for s, c in [(2025, RED), (2026, BLUE)]:
    d = mm[(mm.season == s) & (mm.경기수 >= 20)]
    ax.plot(d.month, d.평균분, marker="o", color=c, lw=2.5, label=f"{s}")
ax.set_xticks(range(3, 11), [f"{i}월" for i in range(3, 11)]); ax.legend(frameon=False); ax.set_ylabel("분", color=SUB)
style(ax, "월별 정규이닝 평균 경기시간", "경기 수 20경기 이상인 달만 표시")
plt.tight_layout(); plt.savefig(f"{OUT}/05_monthly.png", dpi=160); plt.show()

## Step 8. 해석과 한계

**확인된 것**

- 2019~2023년은 같은 투구 수·타자 수 기준 경기시간이 2023년 대비 약 ±1.2분 안에서 평평했습니다. 2024년부터 줄기 시작해 2025년에는 2023년보다 약 9분 짧아졌습니다.
- 2023→2025의 9분 단축은 거의 전부 "투구 1개당 소요시간" 몫입니다. 투구 수 자체는 거의 그대로였습니다.
- 2025→2026에는 경기가 다시 약 3.5분 길어졌는데, 그 대부분(약 3.1분)은 투구 수가 늘어난 몫입니다. 투구 수가 는 이유는 타석당 투구 수가 아니라 타석 수(안타·득점 증가)가 늘어서이고, 볼넷·사구 증가는 그중 3분의 1 정도입니다.
- 2초 단축 뒤 투구 1개당 소요시간은 2025년과 통계적으로 구분되지 않습니다(+0.4분, 95% 구간 약 −0.7 ~ +1.5분). 경기당 0.7분 넘게 줄었다는 근거는 없습니다.

**확인하지 못한 것 (인과 해석의 한계)**

- 투구 1개당 소요시간은 피치클락으로 줄어드는 투구 간격만이 아니라 이닝 교대, 투수 교체, 비디오 판독, 부상·마운드 방문 시간을 모두 담고, 투구 수가 늘면 저절로 조금 내려갑니다. 이 값의 변화를 곧바로 피치클락 효과라고 부를 수 없습니다.
- 2024년에는 ABS(자동 볼 판정), 시프트 제한, 베이스 확대가 피치클락 시범 운영과 함께 들어왔습니다. 2024년 단축분을 피치클락 하나의 효과로 나눌 수 없습니다.
- 2025년에는 연장 11회 축소가 같이 들어왔지만, 이 분석은 정규이닝 경기만 보므로 영향이 작습니다. 견제·투수판 이탈 제한, 판독 규정 등 같은 해의 다른 운영 변화도 함께 들어 있을 수 있습니다.
- 2026년에는 2초 단축과 심판 인터컴이 같은 해에 들어왔고, 타격 지표가 좋아져 타석 수가 늘었습니다. 두 규칙의 효과를 따로 떼어 낼 수 없습니다.
- 비교군이 없습니다. 리그 전체가 동시에 규칙을 바꿨기 때문에, "규칙이 없었다면 몇 분이었을까"는 2019~2023년 수준이 이어졌다는 가정에 기댑니다.
- 신뢰구간은 경기 간 우연한 흔들림만 반영합니다. 어떤 규칙이 무엇을 일으켰는지에 대한 불확실성은 담지 않습니다.
- 경기 중 상황(투수 교체 수, 도루 시도)을 추가로 통제하면 2026-2025 차이가 약 +1.1분으로 커집니다. 다만 이 변수들도 규칙의 영향을 받았을 수 있어 민감도 확인으로만 씁니다.

In [ ]:
res = {"season": {int(k): {kk: round(float(vv), 3) for kk, vv in v.items()} for k, v in m.to_dict("index").items()},
       "decomposition": dec.round(3).to_dict("records"), "regression": reg.round(3).to_dict("records"),
       "slope_sec_per_pitch": round(float(b[-2] * 60), 2), "boot_ci_2526": boot_ci, "reg_2526": reg_2526, "slope_sec_per_batter": round(float(b[-1] * 60), 2),
       "n_games": int(len(games)), "n_reg9": int(len(g9))}
json.dump(res, open(f"{OUT}/results.json", "w"), ensure_ascii=False, indent=1)
print(json.dumps(res["decomposition"], ensure_ascii=False, indent=1))